# Stage 3 - second judge on the MAIN outputs (Kaggle **GPU T4 x2 required**, Internet ON)
Inputs: the generate notebook output (gen_main_seed0.zip, gen_main_seed1.zip). Judge A verdicts come from the repo (`results/judged/main`). 4800 judgments at roughly 2-4 s each = about 3-5 h (guess; judge A took 3.2 h). The smoke cell stops early if the judge's output format is unusable. Paste back the outputs of the last two cells. **Untested.**

In [ ]:
import os, glob, subprocess, torch, json
assert torch.cuda.is_available(), "NO GPU. Settings -> Accelerator -> GPU T4 x2, then restart."
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
TAG = "main"
JUDGE_HF, JUDGE_ID = "NousResearch/Hermes-3-Llama-3.1-8B", "hermes3-8b"   # different model family from judge A (Qwen2.5). Fallback if memory/format problems: "microsoft/Phi-3.5-mini-instruct", "phi35-mini"
GEN_ZIPS = sorted(glob.glob("/kaggle/input/**/gen_main_seed*.zip", recursive=True)); print(GEN_ZIPS)
assert len(GEN_ZIPS) == 2, "add the generate notebook output (gen_main_seed0.zip and gen_main_seed1.zip) as input"
!git clone -q https://github.com/sarrazer24/reins-mini /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!mkdir -p runs
for z in GEN_ZIPS: subprocess.run(f"unzip -qo {z} -d runs", shell=True)
!cp -r results/judged/main/* runs/main/          # judge A verdicts (no responses) committed in the repo
!ls runs/main runs/main/seed0 | head -30
!pip install -q -U transformers accelerate
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS

In [ ]:
# smoke test #2 (after fixing the output format with --json-prefill): 20 outputs, one file.
!rm -f runs/main/seed0/*smoke*
!python -u scripts/11_run_judge.py --run-dir runs/main/seed0 --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID}-smoke2 --hf-model {JUDGE_HF} --json-prefill --only reins.jsonl --limit 20
rows = [json.loads(l) for l in open(f"runs/main/seed0/reins.judged_{JUDGE_ID}-smoke2.jsonl")]
drops = sum(r["verdict"] is None for r in rows)
lab = [r["verdict"]["label"] for r in rows if r["verdict"]]
print("smoke: n", len(rows), "dropped", drops, "harmful label share", sum(lab), "of", len(lab))
assert drops <= 2, "still a format problem: switch to the fallback judge (Phi-3.5-mini) and say so in the write-up"
!rm -f runs/main/seed0/*smoke*

In [ ]:
for d in sorted(glob.glob(f"runs/{TAG}/seed*")):
    print("judging", d, flush=True)
    subprocess.run(f"python -u scripts/11_run_judge.py --run-dir {d} --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID} --hf-model {JUDGE_HF} --json-prefill", shell=True, check=True)

In [ ]:
# criteria J1 (C1-C4 under judge B), then J2/J3 (agreement), then the paired table under judge B
!python scripts/12_evaluate.py --runs runs/{TAG} --judge-id {JUDGE_ID} --size 2B --out results/summary_{TAG}_{JUDGE_ID}.csv
!cp results/criteria.json results/criteria_{JUDGE_ID}.json
!python scripts/16_judge_agreement.py --runs runs/{TAG} --judge-a qwen25-7b --judge-b {JUDGE_ID} --out results/judge_agreement_{JUDGE_ID}.csv
!python scripts/15_paired_analysis.py --runs runs/{TAG} --judge-id {JUDGE_ID} --out results/paired_{TAG}_{JUDGE_ID}.csv | grep -v family

In [ ]:
!find runs/main -name "*.judged_{JUDGE_ID}.jsonl" | zip -q judged_{JUDGE_ID}_main.zip -@
!cp results/*{JUDGE_ID}* . 2>/dev/null; ls -la judged_{JUDGE_ID}_main.zip *{JUDGE_ID}*
!rm -rf /tmp/REINS